# 01 Data Check

This notebook follows the assignment workflow:

**Question → Data Check → EDA → Inference → Conclusion**

Selected variables for this project:
- **Population proportion analysis:** `EverCigaretteUse` with benchmark `p0 = 0.50`
- **Population mean analysis:** `HowMuchDoYouWeighWithoutShoesInKG` with benchmark `mu0 = 68.0`


## 1. Data

In [19]:

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display, Markdown

#圖片解析度
plt.rcParams['figure.dpi'] = 120

#---------------設定整個專案資料夾路徑------------------------
ROOT = Path.cwd().resolve().parent
RAW_PATH = ROOT / "data" / "raw" / "YRBS_2007.csv"
PROCESSED_PATH = ROOT / "data" / "processed" / "yrbs_selected_cleaned.csv"
FIG_DIR = ROOT / "outputs" / "figures"
TAB_DIR = ROOT / "outputs" / "tables"
SUM_DIR = ROOT / "outputs" / "summary"
REF_DIR = ROOT / "references"

for p in [FIG_DIR, TAB_DIR, SUM_DIR, REF_DIR]:
    p.mkdir(parents=True, exist_ok=True)
#---------------設定整個專案資料夾路徑------------------------

#---------------所選的Behavior Variable for Proportion Analysis & Continuous Variable for Mean Analysis------------------------
BEHAVIOR_VAR = "EverCigaretteUse"
BEHAVIOR_P0 = 0.50
CONT_VAR = "HowMuchDoYouWeighWithoutShoesInKG"
CONT_MU0 = 68.0
#---------------所選的Behavior Variable for Proportion Analysis & Continuous Variable for Mean Analysis------------------------

#---------------根據上方BV和CV所需，處理Raw Data做成我們要的資料(Processed Data)並輸出成csv檔案-------------------
def ensure_processed():
    raw = pd.read_csv(RAW_PATH)
    behavior_raw = raw[BEHAVIOR_VAR]
    behavior_clean = behavior_raw.where(behavior_raw.isin([1, 2]))
    #behavior_binary欄位代表處理EverCigaretteUse資料的結果。Success=1;Failure=2;NaN=null -> Success=1;Failure=0;NaN=null
    behavior_binary = pd.Series(
        np.where(behavior_clean == 1, 1, np.where(behavior_clean == 2, 0, np.nan)),
        name="behavior_binary" #
    )
    cont_clean = pd.to_numeric(raw[CONT_VAR], errors="coerce")
    cont_clean = cont_clean.where(cont_clean > 0)
    processed = pd.DataFrame({
        "WhatIsYourSex": raw["WhatIsYourSex"],
        BEHAVIOR_VAR: behavior_raw,
        "behavior_binary": behavior_binary,
        CONT_VAR: cont_clean,
    })
    processed.to_csv(PROCESSED_PATH, index=False)
    return raw, processed
#---------------根據上方BV和CV所需，處理Raw Data做成我們要的資料(Processed Data)並輸出成csv檔案-------------------

raw, processed = ensure_processed()
print("Raw data shape:", raw.shape)
print("Processed data shape:", processed.shape)


Raw data shape: (14041, 103)
Processed data shape: (14041, 4)


## 2. Research Questions

1. **Proportion question:** Is the proportion of students who have ever used cigarettes different from **0.50**?
2. **Mean question:** Is the mean body weight of students different from **68.0 kg**?

## 3. Variable Definition and Coding

### 3.1 Behavior variable
- Variable name: `EverCigaretteUse`
- What it measures: whether a student has ever used cigarettes
- Valid raw codes used in this project: **1** and **2**
- Recoding rule required by the assignment:
  - success = code **1**
  - failure = code **2**
- Missing / invalid values: removed from the analysis

### 3.2 Continuous variable
- Variable name: `HowMuchDoYouWeighWithoutShoesInKG`
- What it measures: body weight without shoes in kilograms
- Valid values used in this project: positive numeric values
- Missing / invalid values: removed from the analysis

In [20]:

required_cols = [BEHAVIOR_VAR, CONT_VAR, "WhatIsYourSex"]
column_check = pd.DataFrame({
    "column": required_cols,
    "exists_in_raw_data": [col in raw.columns for col in required_cols]
})
display(column_check)


,column,exists_in_raw_data
0,EverCigaretteUse,True
1,HowMuchDoYouWeighWithoutShoesInKG,True
2,WhatIsYourSex,True


In [21]:

behavior_raw = raw[BEHAVIOR_VAR]
behavior_raw_freq = (
    behavior_raw.value_counts(dropna=False)
    .sort_index()
    .rename_axis("raw_code")
    .reset_index(name="count")
)
behavior_raw_freq.to_csv(TAB_DIR / "01_behavior_raw_frequency.csv", index=False)
display(behavior_raw_freq)

missing_behavior = int(behavior_raw.isna().sum())
print(f"Missing values in {BEHAVIOR_VAR}: {missing_behavior}")


,raw_code,count
0,1.0,7164
1,2.0,6437
2,NaN,440


Missing values in EverCigaretteUse: 440


**Data check note:** `EverCigaretteUse` contains only raw codes 1 and 2 plus missing values. 
This matches the assignment recoding rule, so no extra invalid codes need to be removed.

In [22]:

processed.head(10)


,WhatIsYourSex,EverCigaretteUse,behavior_binary,HowMuchDoYouWeighWithoutShoesInKG
0,2.0,1.0,1.0,NaN
1,2.0,1.0,1.0,68.04
2,2.0,NaN,NaN,NaN
3,1.0,1.0,1.0,79.38
4,1.0,NaN,NaN,NaN
5,1.0,2.0,0.0,59.42
6,1.0,2.0,0.0,65.77
7,1.0,1.0,1.0,56.70
8,1.0,2.0,0.0,45.36
9,1.0,1.0,1.0,62.60


In [23]:

cont_raw = pd.to_numeric(raw[CONT_VAR], errors="coerce")
continuous_check = pd.DataFrame({
    "metric": [
        "Total rows",
        "Missing values",
        "Non-missing values",
        "Non-positive values",
        "Minimum positive value",
        "Maximum value"
    ],
    "value": [
        len(cont_raw),
        int(cont_raw.isna().sum()),
        int(cont_raw.notna().sum()),
        int((cont_raw.dropna() <= 0).sum()),
        float(cont_raw[cont_raw > 0].min()),
        float(cont_raw.max())
    ]
})
continuous_check.to_csv(TAB_DIR / "01_continuous_data_check.csv", index=False)
display(continuous_check)


,metric,value
0,Total rows,14041.00
1,Missing values,979.00
2,Non-missing values,13062.00
3,Non-positive values,0.00
4,Minimum positive value,34.47
5,Maximum value,180.99


**Data check note:** `HowMuchDoYouWeighWithoutShoesInKG` is numeric, has missing values, and has no non-positive values in the cleaned sample.
The analysis therefore keeps all positive recorded weights and removes missing values only.

In [24]:

variable_definitions_text = f'''# Variable Definitions

## Selected behavior variable
- **Variable name:** `{BEHAVIOR_VAR}`
- **Meaning:** Indicates whether a student has ever used cigarettes.
- **Valid raw codes used in this project:** 1 and 2
- **Project rule:** success = code 1, failure = code 2

## Selected continuous variable
- **Variable name:** `{CONT_VAR}`
- **Meaning:** Student body weight without shoes, measured in kilograms.
- **Valid values used in this project:** positive numeric values
- **Missing handling:** blank / missing values are removed before analysis

## Additional variable used for subgroup EDA
- **Variable name:** `WhatIsYourSex`
- **Role in this project:** used only for optional subgroup EDA
- **Important note:** this project keeps the original sex codes as numeric codes because the code labels are not documented in the assignment PDF.
'''
(REF_DIR / "variable_definitions.md").write_text(variable_definitions_text, encoding="utf-8")

recoding_rules_text = f'''# Recoding Rules

## Behavior variable
- Raw variable: `{BEHAVIOR_VAR}`
- Assignment benchmark: `p0 = {BEHAVIOR_P0:.2f}`
- Recoding used in this project:
  - raw code 1 -> success (1)
  - raw code 2 -> failure (0)
  - anything else / missing -> NA

## Continuous variable
- Raw variable: `{CONT_VAR}`
- Assignment benchmark: `mu0 = {CONT_MU0:.1f}`
- Cleaning used in this project:
  - keep positive numeric values
  - drop missing values
'''
(REF_DIR / "recoding_rules.md").write_text(recoding_rules_text, encoding="utf-8")

print("Processed dataset saved to:", PROCESSED_PATH)
print("Reference files saved in:", REF_DIR)


Processed dataset saved to: C:\Users\asd45\Downloads\project-cycle-2\data\processed\yrbs_selected_cleaned.csv
Reference files saved in: C:\Users\asd45\Downloads\project-cycle-2\references
